In [9]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. crime 데이터 불러오기
df = pd.read_csv("CH_crime_100rows.csv")

# 2. latitude, longitude 결측 제거
df = df.dropna(subset=["Latitude", "Longitude"]).copy()

# 3. geometry 생성
geometry = [Point(xy) for xy in zip(df["Longitude"], df["Latitude"])]

# 4. GeoDataFrame으로 변환
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# 5. ZIP/ZCTA shapefile 불러오기
# 예: tl_2020_us_zcta520.shp 또는 지역 ZIP shapefile
zip_gdf = gpd.read_file("zcta/tl_2020_us_zcta520.shp")

# 6. 좌표계 맞추기
zip_gdf = zip_gdf.to_crs(gdf.crs)

# 7. spatial join으로 각 point에 zipcode 붙이기
joined = gpd.sjoin(gdf, zip_gdf, how="left", predicate="within")

# 8. zipcode 컬럼 확인
# 보통 Census ZCTA 파일에는 ZCTA5CE20 같은 컬럼명이 있음
joined = joined.rename(columns={"ZCTA5CE20": "zipcode"})

# 9. zipcode별 crime count 집계
crime_by_zip = (
    joined.groupby("zipcode")
    .size()
    .reset_index(name="crime_count")
    .sort_values(by="crime_count", ascending=False)
)

# 10. 결과 확인
print(crime_by_zip)

# 11. 저장
crime_by_zip.to_csv("crime_by_zipcode.csv", index=False)

   zipcode  crime_count
10   60616            7
7    60612            6
3    60608            5
23   60637            5
16   60623            5
28   60643            4
24   60639            4
31   60651            4
19   60628            4
12   60618            4
18   60625            3
20   60629            3
17   60624            3
15   60622            3
14   60620            3
13   60619            3
11   60617            3
6    60611            3
35   60659            3
33   60653            2
8    60613            2
30   60649            2
2    60607            2
29   60644            2
26   60641            2
27   60642            2
32   60652            1
34   60654            1
0    60601            1
25   60640            1
22   60636            1
21   60634            1
1    60605            1
9    60614            1
5    60610            1
4    60609            1
36   60706            1


In [10]:
# 1. crime 데이터 불러오기
df = pd.read_csv("CH_crime.csv")

# 2. latitude, longitude 결측 제거
df = df.dropna(subset=["Latitude", "Longitude"]).copy()

# 3. geometry 생성
geometry = [Point(xy) for xy in zip(df["Longitude"], df["Latitude"])]

# 4. GeoDataFrame으로 변환
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# 5. ZIP/ZCTA shapefile 불러오기
# 예: tl_2020_us_zcta520.shp 또는 지역 ZIP shapefile
zip_gdf = gpd.read_file("zcta/tl_2020_us_zcta520.shp")

# 6. 좌표계 맞추기
zip_gdf = zip_gdf.to_crs(gdf.crs)

# 7. spatial join으로 각 point에 zipcode 붙이기
joined = gpd.sjoin(gdf, zip_gdf, how="left", predicate="within")

# 8. zipcode 컬럼 확인
# 보통 Census ZCTA 파일에는 ZCTA5CE20 같은 컬럼명이 있음
joined = joined.rename(columns={"ZCTA5CE20": "zipcode"})

# 9. zipcode별 crime count 집계
crime_by_zip = (
    joined.groupby("zipcode")
    .size()
    .reset_index(name="crime_count")
    .sort_values(by="crime_count", ascending=False)
)

# 10. 결과 확인
print(crime_by_zip)

# 11. 저장
crime_by_zip.to_csv("crime_by_zipcode.csv", index=False)

   zipcode  crime_count
34   60620         6697
33   60619         6584
41   60628         5855
37   60623         5589
31   60617         5517
..     ...          ...
4    60131            6
3    60106            3
5    60176            2
9    60406            2
0    46394            1

[79 rows x 2 columns]
